# 트랜스포머 (Transformer)
2017년 구글이 발표한 논문 "Attention is All You Need"에서 제안된 모델로,<br>
RNN을 사용하지 않고 오직 어텐션만으로 시퀀스를 처리한다.

현재 GPT, BERT 등 대부분의 자연어 처리 모델의 기반이 되는 구조이다.

## RNN vs Transformer

| 구분 | RNN/LSTM | Transformer |
| --- | --- | --- |
| 처리 방식 | 순차적 | 병렬 |
| 학습 속도 | 느림 | 빠름 |
| 장기 의존성 | 약함 | 강함 |
| 위치 정보 | 자연스럽게 처리 | Positional Encoding 필요 |

## 전체 구조
트랜스포머는 **인코더(Encoder)**와 **디코더(Decoder)**로 구성된다.

- **인코더** : 입력 시퀀스를 이해하고 표현으로 변환
- **디코더** : 표현을 받아 출력 시퀀스를 생성

둘 다 N개의 동일한 층을 쌓은 구조다 (논문에서는 N=6).

## 인코더 구조
각 인코더 층은 다음 두 부분으로 구성된다.

1. **Multi-Head Self-Attention** - 입력 내 단어들의 관계 학습
2. **Feed Forward Network** - 비선형 변환

각 부분 뒤에는 **잔차 연결(Residual Connection)**과 **Layer Normalization**이 붙는다.

## 디코더 구조
각 디코더 층은 다음 세 부분으로 구성된다.

1. **Masked Multi-Head Self-Attention** - 미래 토큰을 보지 못하도록 마스킹
2. **Cross-Attention** - 인코더의 출력을 참조
3. **Feed Forward Network**

## Positional Encoding
트랜스포머는 RNN과 달리 순서 정보를 자동으로 학습하지 않는다.<br>
그래서 단어의 **위치 정보**를 직접 더해줘야 한다.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## PyTorch로 트랜스포머 사용
PyTorch에는 트랜스포머가 내장되어 있어 직접 쓸 수 있다.

In [ ]:
import torch
import torch.nn as nn

# 인코더만 사용
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8)
encoder = nn.TransformerEncoder(encoder_layer, num_layers=6)

src = torch.randn(10, 32, 512)  # (seq_len, batch, d_model)
out = encoder(src)
print(out.shape)

In [ ]:
# 전체 트랜스포머 (인코더 + 디코더)
transformer = nn.Transformer(d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6)

src = torch.randn(10, 32, 512)
tgt = torch.randn(20, 32, 512)
out = transformer(src, tgt)
print(out.shape)

## 트랜스포머 기반 모델

| 모델 | 구조 | 용도 |
| --- | --- | --- |
| BERT | 인코더만 | 문장 이해 (분류, 질의응답) |
| GPT | 디코더만 | 문장 생성 |
| T5, BART | 인코더+디코더 | 번역, 요약 |
| ViT | 인코더 (이미지) | 이미지 분류 |
| CLIP | 인코더 (텍스트+이미지) | 멀티모달 |

트랜스포머는 NLP를 넘어 이미지, 음성 등 다양한 분야에 사용되고 있다.

## 핵심 정리
1. **Self-Attention** : 시퀀스 내 모든 위치를 동시에 참조
2. **Multi-Head** : 여러 관점에서 관계를 학습
3. **Positional Encoding** : 순서 정보를 직접 부여
4. **병렬 처리** : RNN보다 학습이 훨씬 빠름
5. **확장성** : 데이터/모델 크기를 키울수록 성능 향상 (LLM의 기반)